[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week07.ipynb)

# Week 7: Regularization & Generalization
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Overfitting; dropout, weight decay, early stopping; basic data augmentation.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Diagnose overfitting from the train and validation gap.
- Apply dropout, weight decay, early stopping, and augmentation.
- Attribute a generalization gain to a specific cause.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. Make a model overfit
Tiny training set, large model: train accuracy goes high, test stays at chance.

In [ ]:
torch.manual_seed(0)
Xtr = torch.randn(30, 20); ytr = torch.randint(0, 2, (30,))   # 30 samples, random labels
Xte = torch.randn(300, 20); yte = torch.randint(0, 2, (300,))

def fit(model, epochs=250, wd=0.0):
    o = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=wd); f = nn.CrossEntropyLoss()
    tr, te = [], []
    for _ in range(epochs):
        model.train(); o.zero_grad(); f(model(Xtr), ytr).backward(); o.step()
        model.eval()
        with torch.no_grad():
            tr.append((model(Xtr).argmax(1) == ytr).float().mean().item())
            te.append((model(Xte).argmax(1) == yte).float().mean().item())
    return tr, te

big = lambda: nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
tr, te = fit(big())
print(f"overfit: train {tr[-1]:.2f} vs test {te[-1]:.2f}")
plt.plot(tr, label="train"); plt.plot(te, label="test"); plt.legend(); plt.title("Overfitting"); plt.show()

## 2. Weight decay and dropout close the gap
Same model, add regularization, watch the gap shrink.

In [ ]:
reg = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Dropout(0.5),
                    nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 2))
tr, te = fit(reg, wd=1e-2)
print(f"regularized: train {tr[-1]:.2f} vs test {te[-1]:.2f}")
plt.plot(tr, label="train"); plt.plot(te, label="test"); plt.legend(); plt.title("With dropout + weight decay"); plt.show()

## 3. Early stopping
Stop when validation stops improving; keep the best checkpoint.

In [ ]:
import copy
model = big(); o = torch.optim.Adam(model.parameters(), 0.01); f = nn.CrossEntropyLoss()
best_acc, best_state, patience, bad = 0.0, None, 30, 0
for epoch in range(250):
    model.train(); o.zero_grad(); f(model(Xtr), ytr).backward(); o.step()
    model.eval()
    with torch.no_grad():
        va = (model(Xte).argmax(1) == yte).float().mean().item()
    if va > best_acc:
        best_acc, best_state, bad = va, copy.deepcopy(model.state_dict()), 0
    else:
        bad += 1
        if bad >= patience:
            print(f"stopped at epoch {epoch}; best val acc {best_acc:.2f}"); break

## 4. Weight decay actually shrinks the weights
Compare the weight norm with and without decay.

In [ ]:
def final_norm(wd):
    torch.manual_seed(0); m = nn.Linear(20, 2); o = torch.optim.Adam(m.parameters(), 0.01, weight_decay=wd); f = nn.CrossEntropyLoss()
    for _ in range(250):
        o.zero_grad(); f(m(Xtr), ytr).backward(); o.step()
    return m.weight.norm().item()
print("weight norm  no decay:", round(final_norm(0.0), 2))
print("weight norm  wd=1e-1 :", round(final_norm(0.1), 2))

## 5. Dropout behaves differently at train vs eval
On in training (random zeros), off at eval (deterministic).

In [ ]:
drop = nn.Dropout(0.5)
x = torch.ones(1, 10)
drop.train(); print("train (random zeros):", drop(x))
drop.eval();  print("eval  (identity)    :", drop(x))

## 6. Capacity vs the gap
Wider models fit the small training set harder; watch the train/test gap grow.

In [ ]:
for width in [4, 16, 128]:
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(20, width), nn.ReLU(), nn.Linear(width, 2))
    tr, te = fit(m)
    print(f"width {width:3d}: train {tr[-1]:.2f}  test {te[-1]:.2f}  gap {tr[-1]-te[-1]:+.2f}")

## 7. Augmentation enlarges the training set
On real images, each epoch sees slightly different inputs (train only).

In [ ]:
from torchvision import transforms
img = torch.rand(3, 32, 32)
aug = transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), transforms.RandomCrop(32, padding=4)])
print("original", tuple(img.shape), "-> augmented", tuple(aug(img).shape))
print("two augmentations differ:", not torch.equal(aug(img), aug(img)))

### Mini-exercise
Add label smoothing (`CrossEntropyLoss(label_smoothing=0.1)`) to the overfitting model and report the train/test gap. Does it help here?

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
torch.manual_seed(0)
m = big(); o = torch.optim.Adam(m.parameters(), 0.01); f = nn.CrossEntropyLoss(label_smoothing=0.1)
for _ in range(250):
    m.train(); o.zero_grad(); f(m(Xtr), ytr).backward(); o.step()
m.eval()
with torch.no_grad():
    tr_a = (m(Xtr).argmax(1) == ytr).float().mean().item()
    te_a = (m(Xte).argmax(1) == yte).float().mean().item()
print(f"label smoothing: train {tr_a:.2f} test {te_a:.2f} (labels are random, so test stays near 0.5)")

**Recap:** watch the train-minus-validation gap, not validation alone; regularize with weight decay, dropout, early stopping, and augmentation; turn dropout and augmentation OFF at eval.

---
Student materials for this week: the lab handout (`labs/week07.html`) and the curated references (`references/week07.html`) in the course site.